# QuantCell Paper Figures

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, average_precision_score, f1_score

os.chdir('C:/Users/wadeb/Desktop/CODEX/data_processing/code')
from codex_project import codex_project
warnings.filterwarnings('ignore')

In [ ]:
annotations_path = 'C:/Users/wadeb/Desktop/annotation_strategies/marker_combos_062525_updated_verified.json'   
base_dir = 'C:/Users/wadeb/Desktop' 
project_name = 'QuantCellPaper'
project_path = f'{base_dir}/{project_name}'
data_path = f'{base_dir}/raw_data'

project_folders = ['122123BW Annotations']
# sample ages in file are correct

sample_labels = {'122123BW Annotations':'QuantCellPaper'}


project_folders_replace = ['012925BW Y1 Annotations', '012925BW Y3 Annoations', '013025BW O1 Annotations', '013125BW O2 Annotations', '013125BW O3 Annotation', '020425BW Y2 Annotations']
sample_labels_replace = {
    '012925BW Y1 Annotations' : 'Y1',
    '012925BW Y3 Annoations' : 'Y3',
    '013025BW O1 Annotations' : 'O1',
    '013125BW O2 Annotations' : 'O2',
    '013125BW O3 Annotation' : 'O3',
    '020425BW Y2 Annotations' : 'Y2'
}

In [ ]:
cell_order = [
    'KLS',
    'MEP',
    'CMP',
    'GMP',
    'CLP',
    'CFU-E',
    'MkP',
    'Erythroblasts',
    'Erythrocytes',
    'Preplatelets',
    'B Cells',
    'CD4 T Cells',
    'CD8 T Cells',
    'cDC',
    'Double Negative T Cells',
    'Monocytes',
    'Neutrophils',
    'Arteries',
    'Capillaries/Sinusoids',
    'Endothelial Niche',
    'Perivascular Niche']


color_order = {
    'KLS' : 'yellowgreen',
    'MEP' : 'palegreen',
    'CMP' : 'orange',
    'GMP' : 'olive',
    'CLP' : 'darkorange',
    'CFU-E' : 'green',
    'MkP' : 'darkseagreen',
    'Erythroblasts' : 'gold',
    'Erythrocytes' : 'wheat',
    'Preplatelets' : 'cornflowerblue',
    'B Cells' : 'violet',
    'CD4 T Cells' : 'red',
    'CD8 T Cells' : 'orangered',
    'cDC' : 'chocolate',
    'Double Negative T Cells' : 'blue',
    'Monocytes' : 'chartreuse',
    'Neutrophils' : 'teal',
    'Arteries' : 'rosybrown',
    'Capillaries/Sinusoids' : 'brown',
    'Endothelial Niche' : 'orange',
    'Perivascular Niche' : 'deepskyblue',
    'Other' : 'white'
}

if os.path.exists(f'{project_path}/label_encoder.joblib'):
    encoder = joblib.load(f'{project_path}/label_encoder.joblib')

In [ ]:


def create_confusion_matrix(conf_matrix, name, normalization='row'):

    if normalization == 'row':
        row_sums = conf_matrix.sum(axis=1) # row sums
        conf_matrix = conf_matrix.divide(row_sums, axis=0) # normalize by row sums
    elif normalization == 'col':
        col_sums = conf_matrix.sum(axis=0) # col sums
        conf_matrix = conf_matrix.divide(col_sums, axis=1) # normalize by col sums
    elif normalization == 'all':
        total_sum = conf_matrix.sum().sum()
        conf_matrix = conf_matrix / total_sum

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(conf_matrix, 
                linecolor='grey', 
                linewidth=0.01, 
                xticklabels=cell_order, 
                yticklabels=cell_order,
                cmap=sns.color_palette("flare", as_cmap=True),
                cbar_kws={'format' :  FuncFormatter(lambda x, _: f'{x:.0f}%'), 'boundaries': np.linspace(0, 1, 100)},)
    ax.collections[0].colorbar.set_ticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.collections[0].colorbar.set_ticklabels(['0%', '20%', '40%', '60%', '80%', '100%'])
    plt.gca().tick_params(labelsize=15, length=6)
    plt.ylabel('Conventional Annotation', fontsize=20)
    #plt.title('% of True Label', fontsize=20)
    #plt.text(-12, -0.5, 'b)',fontsize=20)
    ax.set_xlabel(f'{name}', fontsize=20)    
    ax.set_xticklabels(cell_order, rotation=90)
    return fig, ax

### Fig 1
Voronoi

In [ ]:
original_codex = pd.read_csv(f'{project_path}/codex_conventional_QuantCellPaper.csv', index_col=0)
relabeled_codex = pd.read_csv(f'{project_path}/codex_quantcell_QuantCellPaper.csv', index_col=0)

In [ ]:
X_LIM = (9150, 9257)
Y_LIM = (947, 878)

Marker Expression Annotation Example

In [ ]:
codex = original_codex

marker='CD45'

from scipy.spatial import Voronoi, voronoi_plot_2d, Delaunay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize = (6,4))

section_mask = (codex['x'] > X_LIM[0] - 50) & (codex['x'] < X_LIM[1] + 50) & (codex['y'] > Y_LIM[1] - 50) & (codex['y'] < Y_LIM[0] + 50)
section = codex.loc[section_mask, :].copy()
section.reset_index(inplace=True, drop=True)
vor = Voronoi(section.loc[:, ['x', 'y']])
fig = voronoi_plot_2d(vor, show_vertices=False, show_points=False, line_width=0.05, ax=ax)
axs = fig.axes

section.loc[:, 'color'] = section.loc[:, marker].apply(lambda x: 'grey' if x.count('+') else 'white')

for r in range(len(vor.point_region)):
    region = vor.regions[vor.point_region[r]]
    if not -1 in region:
        polygon = [vor.vertices[i] for i in region]
        plt.fill(*zip(*polygon), color = section.loc[r, 'color'], edgecolor='black')

plt.xlim(X_LIM)
plt.ylim(Y_LIM)

plt.axis('off')
plt.savefig(f'{project_path}/section_voronoi_{marker}_expression.png', dpi=300, bbox_inches='tight', pad_inches=0)

Conventional Annotation

In [ ]:
codex = original_codex

from scipy.spatial import Voronoi, voronoi_plot_2d, Delaunay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize = (6,4))

section_mask = (codex['x'] > X_LIM[0] - 50) & (codex['x'] < X_LIM[1] + 50) & (codex['y'] > Y_LIM[1] - 50) & (codex['y'] < Y_LIM[0] + 50)
section = codex.loc[section_mask, :].copy()
section.reset_index(inplace=True, drop=True)
vor = Voronoi(section.loc[:, ['x', 'y']])
fig = voronoi_plot_2d(vor, show_vertices=False, show_points=False, line_width=0.05, ax=ax)
axs = fig.axes


for r in range(len(vor.point_region)):
    region = vor.regions[vor.point_region[r]]
    if not -1 in region:
        polygon = [vor.vertices[i] for i in region]
        plt.fill(*zip(*polygon), color = color_order[section.loc[r, 'cell_type']], edgecolor='black')

plt.xlim(X_LIM)
plt.ylim(Y_LIM)

plt.axis('off')
plt.savefig(f'{project_path}/section_voronoi_original.png', dpi=300, bbox_inches='tight', pad_inches=0)

QuantCell Annotation

In [ ]:
codex = relabeled_codex

from scipy.spatial import Voronoi, voronoi_plot_2d, Delaunay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize = (6,4))

section_mask = (codex['x'] > X_LIM[0] - 50) & (codex['x'] < X_LIM[1] + 50) & (codex['y'] > Y_LIM[1] - 50) & (codex['y'] < Y_LIM[0] + 50)
section = codex.loc[section_mask, :].copy()
section.reset_index(inplace=True, drop=True)
vor = Voronoi(section.loc[:, ['x', 'y']])
fig = voronoi_plot_2d(vor, show_vertices=False, show_points=False, line_width=0.05, ax=ax)
axs = fig.axes


for r in range(len(vor.point_region)):
    region = vor.regions[vor.point_region[r]]
    if not -1 in region:
        polygon = [vor.vertices[i] for i in region]
        plt.fill(*zip(*polygon), color = color_order[section.loc[r, 'cell_type']], edgecolor='black')

plt.xlim(X_LIM)
plt.ylim(Y_LIM)
plt.axis('off')
plt.savefig(f'{project_path}/section_voronoi_relabeled.png', dpi=300, bbox_inches='tight', pad_inches=0)

### Fig 2


A)

In [ ]:
base_clf_performance_dict=joblib.load(f'{project_path}/base_models/base_clf_performance_dict.joblib')

In [ ]:
axd = plt.figure(figsize=(20, 20)).subplot_mosaic(
    """
    A.B
    C.D
    E..
    """,
    width_ratios=[1, 0.35, 1]
)

plt.subplots_adjust(hspace=0.37)


default_fontsize=22
small_fontsize=14
big_fontsize=30
plt.rcParams['font.size'] = default_fontsize

color_dict = {
    'Nearest Neighbors': '#1f77b4',
    'Multilayer Perceptron': '#ff7f0e',
    'Random Forest': '#2ca02c',
    'Extra Trees': '#d62728',
    'Decision Tree': '#9467bd',
    'Gaussian Naive-Bayes': '#8c564b',
    'Ridge Regression': 'darkcyan',
    "Linear SVC": '#e377c2'
}

names=list(color_dict.keys())


macro_pr_rec = np.zeros((len(names), 1000, 2, 10))

for n, lab in enumerate(names):

    for fold in range(10):

        macro_pr_rec[n, :, 0, fold]=base_clf_performance_dict[lab][2][fold][0]
        macro_pr_rec[n, :, 1, fold]=base_clf_performance_dict[lab][2][fold][1]

num_classes = base_clf_performance_dict[names[0]][4]


for n, lab in enumerate(names):
    axd['A'].plot(np.mean(macro_pr_rec[n, :, 1, :], axis=1), np.mean(macro_pr_rec[n, :, 0, :], axis=1), color=color_dict[lab], label=lab, linewidth=2, alpha=0.6)

axd['A'].set_xlim(0, 1)

axd['A'].set_ylim(0.45, 1)

axd['A'].set_xlabel('Recall', fontsize=default_fontsize)
axd['A'].set_ylabel('Precision', fontsize=default_fontsize)
axd['A'].tick_params(axis='both', which='major', labelsize=default_fontsize)


macro_AP_list = [np.mean([base_clf_performance_dict[lab][0][fold] for fold in range(10)]) for lab in names]

lab_order = np.argsort(macro_AP_list)
macro_AP_list = [macro_AP_list[n] for n in lab_order]
names_sorted = [names[n] for n in lab_order]

min_macro = 1/num_classes

print('Minimum macro-averaged AUPRC:', min_macro)

axd['B'].barh(names_sorted, macro_AP_list, color='white', alpha=1, edgecolor='black')
axd['B'].barh(names_sorted, macro_AP_list, color=[color_dict[names[n]] for n in lab_order], alpha=0.6, edgecolor='black')

#axs[1].set_title('Macro-averaged AUPRC')
axd['B'].axvline(min_macro, color='black', linestyle='--')

axd['B'].spines[['top', 'right']].set_visible(False)


axd['B'].set_xlabel('AUPRC', fontsize=default_fontsize)
axd['B'].tick_params(axis='both', which='major', labelsize=default_fontsize)



import matplotlib.patches as mpatches

results = joblib.load(f'{project_path}/model_selection/top3models_results.joblib')

order = ['RandomForestClassifier', 'MLPClassifier', 'ExtraTreesClassifier']

standard_aps={}
optimized_aps={}

for clf_name in results.keys():
    acc = 0
    score = 0
    y_test_list, y_pred_list, y_proba_list = results[clf_name]
    for n in range(10):
        score += average_precision_score(y_test_list[n], y_proba_list[n], average='macro')
        acc += accuracy_score(y_test_list[n], y_pred_list[n])

    acc /= 10
    score /= 10
    print(f'{clf_name} Macro-averaged AUPRC: {score:.3}')
    print(f'{clf_name} Accuracy: {acc:.3}')
    if 'untuned' in clf_name:
        clf_name = clf_name.split('_untuned')[0]
        standard_aps[clf_name] = score
    else:
        clf_name = clf_name.split('_tuned')[0]
        optimized_aps[clf_name] = score

    
names = ['Random\nForest', 'Multilayer\nPerceptron', 'Extra Trees']

standard_aps = [standard_aps[name] for name in order]
optimized_aps = [optimized_aps[name] for name in order]

axd['C'].bar([0.75, 2.75, 4.75], standard_aps, width=0.75, color='white',zorder=1, alpha=1, edgecolor='black')
axd['C'].bar([0.75, 2.75, 4.75], standard_aps, width=0.75, color=['#2ca02c', '#ff7f0e', '#d62728'],zorder=2, alpha=0.6, edgecolor='black')

axd['C'].bar([1.5, 3.5, 5.5], optimized_aps,width=0.75, color='white',zorder=1, alpha=1, hatch='\\\\',edgecolor='black')
axd['C'].bar([1.5, 3.5, 5.5], optimized_aps,width=0.75, color=['#2ca02c', '#ff7f0e', '#d62728'],zorder=2, alpha=0.6, hatch='\\\\',edgecolor='black')
axd['C'].set_ylim([0.80, 1])

axd['C'].set_ylabel('Average Precision', fontsize=default_fontsize)
axd['C'].spines[['right', 'top']].set_visible(False)
axd['C'].tick_params(axis='both', which='major', labelsize=default_fontsize)

circ1 = mpatches.Patch( facecolor='white',alpha=1,label='Default', edgecolor='black')
circ2= mpatches.Patch( facecolor='white',alpha=1,hatch='\\\\',label='Hyperparameter Tuned', edgecolor='black')
axd['C'].set_xticks([1.125, 3.125, 5.125], names, fontsize=default_fontsize)
axd['C'].set_yticks([0.85, 0.9, 0.95, 1])
axd['C'].set_ylim([0.85, 1])
axd['C'].legend(handles=[circ1, circ2], fontsize=default_fontsize, edgecolor='white', bbox_to_anchor=[1.03, 1.05])



from sklearn.metrics import average_precision_score
import seaborn as sns

stats_loc_dict = joblib.load(f'{project_path}/model_selection/stats_loc_dict_included.joblib')
    

location_order = ['Membrane', 'Cytoplasm', 'Nucleus', 'Cell', 'All']
stat_order = ['Min', 'Max', 'Std.Dev.', 'Median', 'Mean', 'All']

included_average_precision_dict={}
included_average_precision_std_dict={}


for location in location_order:
    included_average_precision_dict[location] = {}
    included_average_precision_std_dict[location] = {}
    for stat in stat_order:

        y_true_dict = stats_loc_dict[location][stat][0]
        y_proba_dict = stats_loc_dict[location][stat][2]

        average_precision_list = []
        for n in y_true_dict.keys():
            average_precision_list.append(average_precision_score(y_true_dict[n], y_proba_dict[n], average='macro'))
        included_average_precision_dict[location][stat] = np.mean(average_precision_list)
        included_average_precision_std_dict[location][stat] = np.std(average_precision_list)

included_df=pd.DataFrame(included_average_precision_dict).T
std_df=pd.DataFrame(included_average_precision_std_dict).T

text_df=included_df.copy()

for row in text_df.index:
    for col in text_df.columns:
            text_df.loc[row, col] = f'{text_df.loc[row, col]:.3f}\n±{std_df.loc[row, col]:.3f}'


sns.heatmap(included_df.loc[location_order, stat_order], 
            annot=text_df.loc[location_order, stat_order], 
            fmt='', 
            cmap='Reds',
            annot_kws={"size": small_fontsize},
            ax=axd['D'])

axd['D'].text(7.7, 3.9, 'Average Precision', rotation=270, fontsize=default_fontsize)
axd['D'].tick_params(labelsize=default_fontsize)
axd['D'].set_xticks(axd['D'].get_xticks(), axd['D'].get_xticklabels(), rotation=90)
axd['D'].set_yticks(axd['D'].get_yticks(), axd['D'].get_yticklabels(), rotation=0)

baseline = included_df.loc['All', 'All']


leave_one_out_dict = joblib.load(f'{project_path}/model_selection/stats_loc_dict_excluded.joblib')



location_order = ['Membrane', 'Cytoplasm', 'Nucleus', 'Cell', 'All']
stat_order = ['Min', 'Max', 'Std.Dev.', 'Median', 'Mean', 'All']

excluded_average_precision_dict={}
excluded_average_precision_std_dict={}


for location in location_order:
    excluded_average_precision_dict[location] = {}
    excluded_average_precision_std_dict[location] = {}
    for stat in stat_order:

        y_true_dict = leave_one_out_dict[location][stat][0]
        y_proba_dict = leave_one_out_dict[location][stat][2]

        average_precision_list = []
        if location == 'All' and stat == 'All':
            excluded_average_precision_dict[location][stat] = None
            excluded_average_precision_std_dict[location][stat] = None
            continue

        for n in y_true_dict.keys():
            average_precision_list.append(average_precision_score(y_true_dict[n], y_proba_dict[n], average='macro'))
        excluded_average_precision_dict[location][stat] = np.mean(average_precision_list)
        excluded_average_precision_std_dict[location][stat] = np.std(average_precision_list)
    
excluded_df=baseline - pd.DataFrame(excluded_average_precision_dict).T
std_df=pd.DataFrame(included_average_precision_std_dict).T

text_df=excluded_df.copy()

for row in text_df.index:
    for col in text_df.columns:
            text_df.loc[row, col] = f'{text_df.loc[row, col]:.3f}\n±{std_df.loc[row, col]:.3f}'



from matplotlib.colors import LinearSegmentedColormap

max_val=np.nanmax(excluded_df.values)
min_val=np.nanmin(excluded_df.values)

high_rgb=(1, 0, 0)
low_rgb=(0, 0, 1)

strength=0.5
pivot=int(np.abs(min_val)/np.abs(max_val-min_val)*256)

colors=np.zeros((256, 3))


colors[:pivot, 0] = 1
colors[pivot:, 0] = np.linspace(1, 1-strength, 256-pivot)
colors[:pivot, 1] = np.linspace(1-strength, 1, pivot)
colors[pivot:, 1] = np.linspace(1, 1-strength, 256-pivot)
colors[:pivot, 2] = np.linspace(1-strength, 1, pivot)
colors[pivot:, 2] = 1


custom_coolwarm_cmap = LinearSegmentedColormap.from_list('Custom_Coolwarm', colors)

sns.heatmap(excluded_df.loc[location_order, stat_order], 
            vmin=min_val,
            vmax=max_val,
            cmap=custom_coolwarm_cmap, 
            annot=text_df.loc[location_order, stat_order],
            fmt='', 
            annot_kws={"size": small_fontsize},
            ax=axd['E'])


axd['E'].tick_params(labelsize=default_fontsize)
axd['E'].set_xticks(axd['E'].get_xticks(), axd['E'].get_xticklabels(),  rotation=90)
axd['E'].set_yticks(axd['E'].get_yticks(), axd['E'].get_yticklabels(), rotation=0)
axd['E'].text(8.2, 4.1, 'Δ Average Precision', rotation=270, fontsize=default_fontsize)


axd['A'].text(-0.2, 1.1, 'A', fontsize=big_fontsize, fontweight='bold', transform=axd['A'].transAxes)
axd['B'].text(-0.2, 1.1, 'B', fontsize=big_fontsize, fontweight='bold', transform=axd['B'].transAxes)
axd['C'].text(-0.2, 1.1, 'C', fontsize=big_fontsize, fontweight='bold', transform=axd['C'].transAxes)
axd['D'].text(-0.2, 1.1, 'D', fontsize=big_fontsize, fontweight='bold', transform=axd['D'].transAxes)
axd['E'].text(-0.2, 1.1, 'E', fontsize=big_fontsize, fontweight='bold', transform=axd['E'].transAxes)



D

E

### Fig 3

In [ ]:


encoder = joblib.load(f'{project_path}/label_encoder.joblib')

quantcell_labels_threshold = pd.read_csv(f'{project_path}/quantcell_labels_95.csv').iloc[:, 0]
quantcell_true = pd.read_csv(f'{project_path}/quantcell_true_labels.csv', index_col=0).loc[:, 'cell_type']
# rows are true labels, columns are predicted labels

confusion_matrix_quantcell_threshold = confusion_matrix(quantcell_true, quantcell_labels_threshold, labels=encoder.classes_)
confusion_matrix_quantcell_threshold = pd.DataFrame(confusion_matrix_quantcell_threshold, 
                              index=encoder.classes_, 
                              columns=encoder.classes_)

confusion_matrix_quantcell_threshold = confusion_matrix_quantcell_threshold.loc[cell_order, cell_order] # reorder rows and columns


create_confusion_matrix(confusion_matrix_quantcell_threshold, 'QuantCell', normalization='col')

In [ ]:
other_mask = quantcell_labels_threshold != 'Other'

In [ ]:
from sklearn.metrics import accuracy_score, precision_score
scores=precision_score(quantcell_true[other_mask], quantcell_labels_threshold[other_mask], average=None)

In [ ]:
min(scores)

B

In [ ]:
fraction_annotated = joblib.load(f'{project_path}/fraction_annotated.joblib')

In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

base = fraction_annotated['Conventional']
labels = ['Conventional\nAnnotation', 'FDR < 1%', 'FDR < 5%', 'FDR < 10%']

ax.bar(range(len(labels)), 
       height=base, 
       color='white', 
       alpha=1,
       edgecolor='black')

ax.bar(range(len(labels)), 
       height=base, 
       color='blue', 
       alpha=0.5,
       edgecolor='black')

ax.bar(range(len(labels)), 
       height=[0] + [fraction_annotated[x] for x in labels[1:]], 
       bottom=base, 
       color='white', 
       alpha=1,
       edgecolor='black')

ax.bar(range(len(labels)), 
       height=[0] + [fraction_annotated[x] for x in labels[1:]], 
       bottom=base, 
       color='green', 
       alpha=0.5,
       edgecolor='black')

green_patch = mpatches.Patch(facecolor=(0,0.5,0,0.5), label='QuantCell', edgecolor=(0,0,0,1))
blue_patch = mpatches.Patch(facecolor=(0,0,1,0.5), label='Conventional Annotation', edgecolor=(0,0,0,1))

fig.legend(handles=[blue_patch, green_patch], fontsize=20, bbox_to_anchor=[0.64, 0.92], frameon=False)
ax.set_ylim([0, 1])
ax.set_xticks(range(len(labels)),  labels=labels)
ax.set_ylabel('Fraction Annotated', fontsize=20)
ax.spines[['right', 'top']].set_visible(False)
ax.tick_params(labelsize=20)

In [ ]:
0.5703832426470887/0.3312075989239741 * 100

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
xvals = [float(x.split(' ')[2][:-1]) for x in fraction_annotated.keys() if 'FDR' in x ]
yvals = [fraction_annotated[x] for x in fraction_annotated.keys() if 'FDR' in x ]
ax.plot(xvals, yvals, color='blue')
ax.scatter(xvals, yvals, color='blue', s=100, edgecolor='black', zorder=3)
ax.spines[['top', 'right']].set_visible(False)

ax.set_xlabel('FDR Threshold (%)', fontsize=20)
ax.set_ylabel('Fraction Annotated', fontsize=20)
ax.tick_params(labelsize=20)
ax.set_xlim([0, 10])

C

In [ ]:
original_codex = pd.read_csv(f'{project_path}/codex_conventional_QuantCellPaper.csv', index_col=0)
relabeled_codex = pd.read_csv(f'{project_path}/codex_quantcell_QuantCellPaper.csv', index_col=0)

original_frequencies = original_codex.loc[:, 'cell_type'].value_counts()/len(original_codex)
relabeled_frequencies = relabeled_codex.loc[:, 'cell_type'].value_counts()/len(relabeled_codex)

In [ ]:
from scipy.stats import pearsonr
fig, ax = plt.subplots()

xvals= original_frequencies
yvals= relabeled_frequencies
index = yvals.index[yvals.index != 'Other']
xvals = xvals[xvals.index.isin(index)]
yvals = yvals[yvals.index.isin(index)]
plt.scatter(xvals, yvals, s=20, color='black')
plt.xlabel('Conventional Annotation Frequency', fontsize=20)
plt.ylabel('QuantCell Frequency', fontsize=20)
ax.tick_params(labelsize=20)
r, p = pearsonr(xvals, yvals)
ax.set_title(f'r={str(r)[:5]}   p-Value={str(p)[:3] + str(p)[-4:]}', fontsize=20)
ax.spines[['top', 'right']].set_visible(False)

a, b = np.polyfit(xvals.sort_values(), yvals[xvals.sort_values().index], 1)
plt.plot(xvals.sort_values(), a * xvals.sort_values() + b, color='grey', linestyle='--')

plt.xlim([0, 0.12])
plt.ylim([-0.005, None])

In [ ]:
facs_data = pd.read_csv(f'{data_path}/OldYoungRound1FACS.csv', index_col=0)

facs_data.loc[:, 'Mouse ID'] = ['O1', 'O2', 'O3', 'Y1', 'Y2', 'Y3']
facs_data.set_index('Mouse ID', inplace=True, drop=True)

for x in facs_data.columns:
    facs_data.rename(columns={x: x.split(' ')[0]}, inplace=True)
    x= x.split(' ')[0]

    facs_data.rename(columns={x: x.split('+')[0]}, inplace=True)
    x= x.split('+')[0]
    if x in ['T', 'B']:
        facs_data.rename(columns={x: x + ' Cells'}, inplace=True)

    if x in ['CD4', 'CD8']:
        facs_data.rename(columns={x: x + ' T Cells'}, inplace=True)
    
facs_data.loc[:, 'MPP'] = facs_data.loc[:, 'KLS'] - facs_data.loc[:, 'HSC']
facs_data.drop(columns=['Granulocytes', 'FLK2', 'FLK2-', 'HSC', 'Progenitor', 'Lineage-', 'MPP', 'T Cells'], inplace=True)
facs_data /= 100

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))


cells_of_interest = [x for x in cell_order if x in facs_data.columns and x in original_frequencies.index and x in relabeled_frequencies.index]

size=200
offset=0.15

ax.scatter(0, facs_data.loc[:, cells_of_interest[0]].mean(), c='red', marker='*', label='FACS BM Percentage', s=1.5*size, alpha=0.6)
ax.scatter(0+offset, original_frequencies[cells_of_interest[0]], c='blue', marker='^', label='Conventional Annotation', s=size, alpha=1)
ax.scatter(0+2*offset, relabeled_frequencies[cells_of_interest[0]], c='green', marker='s', label='QuantCell Annotation', s=size, alpha=0.6)

for n, ct in enumerate(cells_of_interest[1:]):
    ax.scatter(n+1, facs_data.loc[:, ct].mean(), c='red', marker='*', s=1.5*size, alpha=0.6)
    ax.scatter(n+1+offset, original_frequencies[ct], c='blue', marker='^', s=size, alpha=1)
    ax.scatter(n+1+2*offset, relabeled_frequencies[ct], c='green', marker='s', s=size, alpha=0.6)

ax.legend(fontsize=20, frameon=False, loc='upper left')

formatted_labels = cells_of_interest.copy()
formatted_labels = [x.replace(' T ', '\nT ') for x in formatted_labels]

ax.set_xticks(range(len(formatted_labels)), formatted_labels, fontsize=20)
plt.tick_params(labelsize=20)
ax.spines[['top', 'right']].set_visible(False)
plt.ylabel('Frequency of Labeled BM Cells', fontsize=20)


E

In [ ]:
annospat_labels = pd.read_csv(f'{project_path}/AnnoSpat/outputdir_threshold/trte_labels_ELM_IMC_T1D_AnnoSpat.csv', index_col=0).loc[:, 'label']
annospat_true = pd.read_csv(f'{project_path}/AnnoSpat/annospat_true_labels.csv', index_col=0).loc[:, 'cell_type']

maps_labels = pd.read_csv(f'{project_path}/MAPS/results/cell_phenotyping/pred_labels_all_features.csv', index_col=0).iloc[:, 0]
maps_true = pd.read_csv(f'{project_path}/MAPS/data/cell_phenotyping/test_all_features.csv', index_col=0).loc[:, 'cell_label']

maps_labels = encoder.inverse_transform(maps_labels)
maps_true = encoder.inverse_transform(maps_true)

quantcell_labels_threshold = pd.read_csv(f'{project_path}/quantcell_labels_95.csv').iloc[:, 0]
quantcell_labels_no_threshold = pd.read_csv(f'{project_path}/quantcell_labels_no_threshold.csv').iloc[:, 0]
quantcell_true = pd.read_csv(f'{project_path}/quantcell_true_labels.csv', index_col=0).loc[:, 'cell_type']

astir_labels = pd.read_csv(f'{project_path}/Astir/astir_labels.csv', index_col=0).loc[:, 'cell_type']
astir_true = pd.read_csv(f'{project_path}/Astir/astir_true_labels.csv').loc[:, 'cell_type']

annospat = accuracy_score(annospat_true, annospat_labels)
maps = accuracy_score(maps_true, maps_labels)
quantcell_no_threshold = accuracy_score(quantcell_true, quantcell_labels_no_threshold)
quantcell_other_mask = quantcell_labels_threshold != 'Other'
quantcell_threshold = accuracy_score(quantcell_true[quantcell_other_mask], quantcell_labels_threshold[quantcell_other_mask])
astir = accuracy_score(astir_true, astir_labels)


fig, ax = plt.subplots(figsize=(10, 7))

ax.bar(['Astir', 'AnnoSpat', 'MAPS', 'QuantCell\nw/o\nThresholding', 'QuantCell'], 
       [astir, annospat, maps, quantcell_no_threshold, quantcell_threshold], 
       color='white', 
       edgecolor='black',
       alpha=1)    

ax.bar(['Astir', 'AnnoSpat', 'MAPS', 'QuantCell\nw/o\nThresholding', 'QuantCell'], 
       [astir, annospat, maps, quantcell_no_threshold, quantcell_threshold], 
       color=['crimson', 'purple', 'orange', 'teal', 'green'], 
       edgecolor='black',
       alpha=0.5)



ax.set_ylabel('Accuracy', fontsize=20)
ax.tick_params(labelsize=20)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1], fontsize=20)
plt.tight_layout()
ax.spines[['top', 'right']].set_visible(False)

for i, score in enumerate([astir, annospat, maps, quantcell_no_threshold, quantcell_threshold]):
    ax.text(i-0.23, score + 0.01, f'{score:.3f}', fontsize=20, color='black')

### Fig 4

A

In [ ]:
from sklearn.metrics import f1_score
results = joblib.load(f'{project_path}/model_selection/top3models_results.joblib')
encoder = joblib.load(f'{project_path}/label_encoder.joblib')

In [ ]:
y_test, y_pred, y_proba=results['RandomForestClassifier_tuned']
score_matrix = np.zeros((len(encoder.classes_), 10))
for n in range(10):
    score_matrix[:, n] = f1_score(y_test[n], y_pred[n], average=None)

f1_scores = {}
for i, label in enumerate(encoder.classes_):
    f1_scores[label] = np.mean(score_matrix[i, :])

In [ ]:
from quantcell import quantcell_project

base_dir = '/store/Projects/wboohar/PhenoCycler/' 
project_name = 'QuantCellPaper'

project = quantcell_project()
project.initialize(base_path=base_dir, project_name=project_name)

In [ ]:
from codex_project import read_marker_combos

frequency_dict={}
size_dict={}
number_marker_dict={}
circularity_dict={}

marker_combos = read_marker_combos(annotations_path)

total_annotated = project.codex.loc[:, 'cell_type'].value_counts().sum() - project.codex.loc[:, 'cell_type'].value_counts()['Other']

for ct in cell_order:
    mask = project.codex.loc[:, 'cell_type'] == ct
    frequency_dict[ct] = project.codex.loc[mask, 'cell_type'].value_counts()[ct] / total_annotated
    size_dict[ct] = project.codex.loc[mask, 'Cell: Area µm^2'].mean()
    circularity_dict[ct] = project.codex.loc[mask, 'Cell: Circularity'].mean()
    number_marker_dict[ct] = len(marker_combos[ct])
    

In [ ]:
from scipy.stats import pearsonr

fig, axs = plt.subplots(2, 2, sharey=True, figsize=(14,7))
axs=axs.flatten()

axs[0].scatter([frequency_dict[ct] for ct in cell_order], [f1_scores[ct] for ct in cell_order], color='black')
axs[0].set_xlabel('Cell Frequency', fontsize=15)
axs[0].set_ylabel('F1 Score', fontsize=15)
axs[0].spines[['top', 'right']].set_visible(False)
axs[0].tick_params(labelsize=15)
axs[0].set_xlim(0, 0.1)
axs[0].set_ylim(0.68, 1)
stat, pvalue = pearsonr([frequency_dict[ct] for ct in cell_order], [f1_scores[ct] for ct in cell_order])
axs[0].text(0.0334, 1.02, f'r= {stat:.2f}       p={pvalue:.3f}', fontsize=15)

axs[1].scatter([size_dict[ct] for ct in cell_order], [f1_scores[ct] for ct in cell_order], color='black')
axs[1].set_xlabel('Cell Size (µm²)', fontsize=15)
axs[1].spines[['top', 'right']].set_visible(False)
axs[1].tick_params(labelsize=15)
stat, pvalue = pearsonr([size_dict[ct] for ct in cell_order], [f1_scores[ct] for ct in cell_order])
axs[1].text(34.6, 1.02, f'r= {stat:.2f}       p={pvalue:.3f}', fontsize=15)

axs[2].scatter([circularity_dict[ct] for ct in cell_order], [f1_scores[ct] for ct in cell_order], color='black')
axs[2].set_xlabel('Cell Circularity', fontsize=15)
axs[2].set_ylabel('F1 Score', fontsize=15)
axs[2].spines[['top', 'right']].set_visible(False)
axs[2].tick_params(labelsize=15)
stat, pvalue = pearsonr([circularity_dict[ct] for ct in cell_order], [f1_scores[ct] for ct in cell_order])
axs[2].text(0.8535, 1.02, f'r= {stat:.2f}       p={pvalue:.3f}', fontsize=15)

axs[3].scatter([number_marker_dict[ct] for ct in cell_order], [f1_scores[ct] for ct in cell_order], color='black')
axs[3].set_xlabel('Number of Markers', fontsize=15)
axs[3].spines[['top', 'right']].set_visible(False)
axs[3].tick_params(labelsize=15)
stat, pvalue = pearsonr([number_marker_dict[ct] for ct in cell_order], [f1_scores[ct] for ct in cell_order])
axs[3].text(5.32, 1.02, f'r= {stat:.2f}       p={pvalue:.3f}', fontsize=15)

plt.subplots_adjust(hspace=0.6, wspace=0.2)



4B

In [ ]:
gene_ko_dict = joblib.load(f'{project_path}/model_selection/gene_ko_dict.joblib')
tuned_predictions = joblib.load(f'{project_path}/model_selection/top3models_results.joblib')['RandomForestClassifier_tuned']
gene_order = sorted(gene_ko_dict.keys(), key=str.lower)
f1_baseline = np.mean([f1_score(tuned_predictions[0][n], tuned_predictions[1][n], average=None) for n in range(10)], axis=0)

In [ ]:
f1_matrix = np.zeros((len(gene_order), 10, len(encoder.classes_)))
for n, gene in enumerate(gene_order):
    for fold in range(10):
        y_true = gene_ko_dict[gene][0][fold]
        y_pred = gene_ko_dict[gene][1][fold]
        f1_matrix[n, fold, :] = f1_score(y_true, y_pred, average=None)


In [ ]:
long_df = pd.DataFrame(columns=['Gene', 'Fold', 'Cell Type', 'F1 Score'])
for n, gene in enumerate(gene_order):
    for fold in range(10):
        for cell_type in encoder.classes_:
            cell_index = encoder.transform([cell_type])[0]
            long_df = long_df.append({
                'Gene': gene,
                'Fold': fold,
                'Cell Type': cell_type,
                'F1 Score': f1_matrix[n, fold, cell_index] - f1_baseline[cell_index]
            }, ignore_index=True)

In [ ]:
from codex_project import read_marker_combos
marker_combos=read_marker_combos(annotations_path)

In [ ]:
for ct in ['CD8 T Cells']:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    sns.swarmplot(data=long_df[long_df['Cell Type'] == ct],
                   x='Gene',
                   y='F1 Score',
                   ax=ax,
                   color='black',
                   size=4)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_title(f'{ct}', fontsize=20)
    plt.axhline(0, color='black', linestyle='--', linewidth=2)
    ax.set_ylabel('Δ F1 Score', fontsize=20)
    ax.set_xlabel('Gene Excluded', fontsize=20)
    ax.set_xticklabels([' ']*len(gene_order), fontsize=20)
    ax.tick_params(labelsize=20)
    mean_std = long_df.groupby(['Gene', 'Cell Type']).std().groupby('Cell Type').mean().loc[ct]
    offset=-1
    ylim=0
    for n, gene in enumerate(gene_order):
       
        mean_difference = long_df[(long_df['Gene'] == gene) & (long_df['Cell Type'] == ct)]['F1 Score'].mean()
        min_difference = long_df[(long_df['Gene'] == gene) & (long_df['Cell Type'] == ct)]['F1 Score'].min()

        if np.abs(mean_difference) > 3*mean_std.values[0]:
            if n < 5:
                offset = 1
            if n > len(gene_order) - 12:
                offset = 1
            ax.text(gene_order.index(gene) + 1.6*offset + 0.20*len(gene)*offset, 1.1*min_difference, gene, fontsize=20, ha='center', va='bottom')
            offset *= -1
        if min_difference < ylim:
            ylim = min_difference
    ax.set_ylim([1.2*ylim, None])
    
    

## S1
A

In [ ]:

base_model_names = [
    'Nearest Neighbors',
    'Multilayer Perceptron',
    'Random Forest',
    'Extra Trees',
    'Decision Tree',
    'Gaussian Naive-Bayes',
    'Ridge Regression',
    "Linear SVC"
]

color_dict = {
    'Nearest Neighbors': '#1f77b4',
    'Multilayer Perceptron': '#ff7f0e',
    'Random Forest': '#2ca02c',
    'Extra Trees': '#d62728',
    'Decision Tree': '#9467bd',
    'Gaussian Naive-Bayes': '#8c564b',
    'Ridge Regression': 'darkcyan',
    "Linear SVC": '#e377c2'
}

fig,ax = plt.subplots(figsize=(8, 6))


if not 'names_sorted' in globals():
    names_sorted = ['Random Forest', 'Multilayer Perceptron', 'Extra Trees', 'Linear SVC', 'Nearest Neighbors', 'Decision Tree', 'Ridge Regression', 'Gaussian Naive-Bayes']

for n, model in enumerate(names_sorted):
    time_elapsed = joblib.load(f'{project_path}/base_models/{model}_time_elapsed.joblib')

    time_elapsed = time_elapsed / 10 # average over 10 folds

    print(f'{model} time elapsed: {time_elapsed:.2f} seconds')
    ax.barh(n, time_elapsed, color='white', label=model, alpha=1, edgecolor='black')
    ax.barh(n, time_elapsed, color=color_dict[model], label=model, alpha=0.6, edgecolor='black')
    ax.text(time_elapsed*1.05, n, f'{time_elapsed:.2f}', va='center', fontsize=22)
ax.set_xscale('log')
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlabel('Training Time (seconds)', fontsize=22)
ax.set_yticks(range(len(names_sorted)), names_sorted, fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.set_xticks([1, 10, 100, 1000], [1, 10, 100, 1000], fontsize=22)


## S2

### QuantCell Threshold

In [ ]:
# rows are true labels, columns are predicted labels
quantcell_labels_threshold = pd.read_csv(f'{project_path}/quantcell_labels_95.csv').iloc[:, 0]
quantcell_true = pd.read_csv(f'{project_path}/quantcell_true_labels.csv', index_col=0).loc[:, 'cell_type']

confusion_matrix_quantcell_threshold = confusion_matrix(quantcell_true, quantcell_labels_threshold, labels=encoder.classes_)
confusion_matrix_quantcell_threshold = pd.DataFrame(confusion_matrix_quantcell_threshold, 
                              index=encoder.classes_, 
                              columns=encoder.classes_)

confusion_matrix_quantcell_threshold = confusion_matrix_quantcell_threshold.loc[cell_order, cell_order] # reorder rows and columns

create_confusion_matrix(confusion_matrix_quantcell_threshold, 'QuantCell w/ Threshold', normalization='col')


### QuantCell No Threshold

In [ ]:
# rows are true labels, columns are predicted labels
quantcell_labels_no_threshold = pd.read_csv(f'{project_path}/quantcell_labels_no_threshold.csv').iloc[:, 0]
quantcell_true = pd.read_csv(f'{project_path}/quantcell_true_labels.csv', index_col=0).loc[:, 'cell_type']

confusion_matrix_quantcell_no_threshold = confusion_matrix(quantcell_true, quantcell_labels_no_threshold, labels=encoder.classes_)
confusion_matrix_quantcell_no_threshold = pd.DataFrame(confusion_matrix_quantcell_no_threshold, 
                              index=encoder.classes_, 
                              columns=encoder.classes_)

confusion_matrix_quantcell_no_threshold = confusion_matrix_quantcell_no_threshold.loc[cell_order, cell_order] # reorder rows and columns

create_confusion_matrix(confusion_matrix_quantcell_no_threshold, 'QuantCell w/o Threshold', normalization='col')


### AnnoSpat

In [ ]:
annospat_labels = pd.read_csv(f'{project_path}/AnnoSpat/outputdir_threshold/trte_labels_ELM_IMC_T1D_AnnoSpat.csv', index_col=0).loc[:, 'label']
annospat_true = pd.read_csv(f'{project_path}/AnnoSpat/annospat_true_labels.csv', index_col=0).loc[:, 'cell_type']

confusion_matrix_annospat = confusion_matrix(annospat_true, annospat_labels, labels=encoder.classes_)
confusion_matrix_annospat = pd.DataFrame(confusion_matrix_annospat, 
                              index=encoder.classes_, 
                              columns=encoder.classes_)

confusion_matrix_annospat = confusion_matrix_annospat.loc[cell_order, cell_order] # reorder rows and columns

create_confusion_matrix(confusion_matrix_annospat, 'AnnoSpat', normalization='col')


### MAPS

In [ ]:
maps_labels = pd.read_csv(f'{project_path}/MAPS/results/cell_phenotyping/pred_labels_all_features.csv', index_col=0).iloc[:, 0]
maps_true = pd.read_csv(f'{project_path}/MAPS/data/cell_phenotyping/test_all_features.csv', index_col=0).loc[:, 'cell_label']

maps_labels = encoder.inverse_transform(maps_labels)
maps_true = encoder.inverse_transform(maps_true)

confusion_matrix_maps = confusion_matrix(maps_true, maps_labels, labels=encoder.classes_)
confusion_matrix_maps = pd.DataFrame(confusion_matrix_maps, 
                              index=encoder.classes_, 
                              columns=encoder.classes_)

confusion_matrix_maps = confusion_matrix_maps.loc[cell_order, cell_order] # reorder rows and columns
create_confusion_matrix(confusion_matrix_maps, 'MAPS', normalization='col')

### Astir

In [ ]:
astir_labels = pd.read_csv(f'{project_path}/Astir/astir_labels.csv', index_col=0).loc[:, 'cell_type']
astir_true = pd.read_csv(f'{project_path}/Astir/astir_true_labels.csv').loc[:, 'cell_type']

confusion_matrix_astir = confusion_matrix(astir_true, astir_labels, labels=(encoder.classes_))
confusion_matrix_astir = pd.DataFrame(confusion_matrix_astir, 
                              index=encoder.classes_, 
                              columns=encoder.classes_)

confusion_matrix_astir = confusion_matrix_astir.loc[cell_order, cell_order] # reorder rows and columns

create_confusion_matrix(confusion_matrix_astir, 'Astir', normalization='col')

In [ ]:


fig,ax = plt.subplots(figsize=(8, 6))

names_sorted = ['Astir', 'AnnoSpat', 'MAPS', 'QuantCell w/o\nThresholding', 'QuantCell']

astir_time = pd.read_csv(f'{project_path}/Astir/astir_time.csv', index_col=0).iloc[0, 0]
with open(f'{project_path}/AnnoSpat/time_elapsed_threshold.txt', 'r') as f:
    annospat_time = float(f.read().strip())/10**9 # convert from nanoseconds to seconds
maps_time = pd.read_csv(f'{project_path}/MAPS/results/cell_phenotyping/time_elapsed_maps_all_features.csv', index_col=0).iloc[0, 0]
quantcell_threshold_time = pd.read_csv(f'{project_path}/time_elapsed_quantcell_threshold.csv').iloc[0, 0]
quantcell_no_threshold_time = pd.read_csv(f'{project_path}/time_elapsed_quantcell_no_threshold.csv').iloc[0, 0]


ax.barh(range(len(names_sorted)),
        [astir_time, annospat_time, maps_time, quantcell_no_threshold_time,quantcell_threshold_time],
        color='white', label='Training Time', alpha=1, edgecolor='black')
ax.barh(range(len(names_sorted)),
        [astir_time, annospat_time, maps_time, quantcell_no_threshold_time,quantcell_threshold_time],
        color=['crimson', 'purple', 'orange', 'teal', 'green'], label='Training Time', alpha=0.5, edgecolor='black')
ax.set_xscale('log')
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlabel('Annotation Runtime (seconds)', fontsize=22)
ax.set_yticks(range(len(names_sorted)), names_sorted, fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.set_xticks([1, 10, 100, 1000, 10000], [1, 10, 100, 1000, 10000], fontsize=22)
ax.text(astir_time*1.1, 0, f'{astir_time:.2f}', fontsize=22, color='black', va='center')
ax.text(annospat_time*1.1, 1, f'{annospat_time:.2f}', fontsize=22, color='black', va='center')
ax.text(maps_time*1.1, 2, f'{maps_time:.2f}', fontsize=22, color='black', va='center')
ax.text(quantcell_no_threshold_time*1.1, 3, f'{quantcell_no_threshold_time:.2f}', fontsize=22, color='black', va='center')
ax.text(quantcell_threshold_time*1.1, 4, f'{quantcell_threshold_time:.2f}', fontsize=22, color='black', va='center')


In [ ]:
pd.read_csv(f'{project_path}/MAPS/results/cell_phenotyping/time_elapsed_maps.csv', index_col=0).iloc[0, 0]

In [ ]:
astir_time

In [ ]:
maps_time